# Building a Simple Neural Network with PyTorch

In [1]:
import torch
print(torch.__version__)

2.8.0


In [2]:
import torch.nn as nn
import torch.optim as optim

# Setting up seed ensures results are reproducible and consistent across runs
torch.manual_seed(25)

## Step 1: Data Preparation

We'll create a simple synthetic dataset representing the relationship between house size and price.

In [ ]:
# Input features: house sizes (in 100 sqft units, e.g., 5.0 = 500 sqft)
size = torch.tensor([[5.0], [7.50], [10.0], [12.50], [15.00], [17.50], [20.00]], dtype=torch.float32)

# Target values: house prices (in $100,000 units, e.g., 1.50 = $150,000)
price = torch.tensor([[1.50], [2.00], [2.50], [3.00], [3.50], [4.00], [4.50]], dtype=torch.float32)


## Step 2: Define the Neural Network Model

We'll use `nn.Sequential` with a single `nn.Linear(1, 1)` layer:
- `nn.Linear(1, 1)`: 1 input feature → 1 output value
- The layer automatically manages weights and biases that will be learned during training

In [ ]:
# Create a simple neural network model with one input and one output
model = nn.Sequential(
    nn.Linear(1, 1)  # One input feature (size) and one output (price)
)
model

Sequential(
  (0): Linear(in_features=1, out_features=1, bias=True)
)

In [5]:
model[0].weight, model[0].bias

(Parameter containing:
 tensor([[0.5036]], requires_grad=True),
 Parameter containing:
 tensor([-0.6142], requires_grad=True))

## Step 3: Define Loss Function and Optimizer

- **Loss Function**: Measures the difference between predictions and actual values
- **Optimizer**: Updates model parameters (weights and biases) to minimize the loss
    - **Learning Rate**: Controls how much the optimizer adjusts parameters in each step

In [ ]:
loss_fn = nn.MSELoss()  # Mean Squared Error Loss
optimizer = optim.SGD(model.parameters(), lr=0.001)  # Stochastic Gradient Descent Optimizer

## Step 4: Training Loop

The training loop repeats the following steps for multiple epochs:
0. Reset Gradients. Clears gradients from precious epochs.
1. Forward pass: Make predictions
2. Calculate loss: Compare predictions with targets
3. Backward pass: Compute gradients
4. Update parameters: Adjust weights and bias

In [ ]:
for epoch in range(500):
    # Reset gradients to zero (gradients accumulate by default)
    optimizer.zero_grad()

    # Forward pass: Make predictions
    predicted_price = model(size)
    
    # Calculate loss: Compare predictions with actual prices
    loss = loss_fn(predicted_price, price)

    # Backward pass: Compute gradients
    loss.backward()

    # Update model parameters using computed gradients
    optimizer.step()

    # Print progress every 50 epochs
    if (epoch+1) % 50 == 0:
        print(f'Epoch [{epoch+1}/500] | Loss: {loss.item():.4f}')

Epoch [50/500] | Loss: 0.1714
Epoch [100/500] | Loss: 0.1667
Epoch [150/500] | Loss: 0.1622
Epoch [200/500] | Loss: 0.1578
Epoch [250/500] | Loss: 0.1536
Epoch [300/500] | Loss: 0.1494
Epoch [350/500] | Loss: 0.1454
Epoch [400/500] | Loss: 0.1414
Epoch [450/500] | Loss: 0.1376
Epoch [500/500] | Loss: 0.1339


### Inspecting Learned Parameters

After training, check the learned weight and bias values.

In [ ]:
layers = model[0]
print(f'Learned parameters: Weight = {layers.weight.item():.4f}, Bias = {layers.bias.item():.4f}')

Learned parameters: Weight = 0.2680, Bias = -0.4849


## Step 5: Making Predictions

Use the trained model to predict prices for new house sizes. Use `torch.no_grad()` to disable gradient computation during inference (saves memory and computation).

In [ ]:
# Create input tensor for prediction (14.0 = 1400 sqft)
house_size = torch.tensor([[14.0]], dtype=torch.float32)

In [ ]:
# Make prediction without computing gradients (faster and uses less memory)
with torch.no_grad():
    predicted_price = model(house_size)
    # Convert back to original units for display
    print(f'Predicted price for a house of size {house_size.item()*100} sqft is ${predicted_price.item()*100000:.2f}')

Predicted price for a house of size 1400.0 sqft is $326676.06
